In [2]:
from pathlib import Path
import pandas as pd
meteo_dir = Path("宝天曼站气象数据（2017-2018）")
years = range(2017, 2019)
meteo_frames = []
for year in years:
    file_path = meteo_dir / f"{year}年宝天曼气象30分钟数据.xls"
    df_year = pd.read_excel(file_path)
    df_year = df_year.iloc[1:].reset_index(drop=True)
    df_year["来源年份"] = year
    meteo_frames.append(df_year)
df_meteo_30min = pd.concat(meteo_frames, ignore_index=True)
df_meteo_30min.head()

,年,月,日,时,分,秒,近地面空气温度,冠层上方空气温度,近地面空气湿度,冠层上方空气湿度,...,向下的长波辐射,向上的长波辐射,净辐射,一层土壤温度,二层土壤温度,一层土壤体积含水量,二层土壤体积含水量,三层土壤体积含水量,降水量,来源年份
0,2017,1,1,0,30,0,3.24,4.09,57.43,52.44,...,312.8,325.7,-13.76,1.45,2.99,0.1948,0.1779,0.2339,0,2017
1,2017,1,1,1,0,0,2.85,4.05,57.14,52.19,...,316,325.6,-10.34,1.46,2.99,0.1949,0.1779,0.2338,0,2017
2,2017,1,1,1,30,0,2.83,3.78,56.24,52.98,...,317,325.1,-8.48,1.47,2.99,0.1952,0.178,0.2337,0,2017
3,2017,1,1,2,0,0,2.76,3.55,58.67,56.45,...,320.2,324.7,-4.82,1.48,2.99,0.1954,0.1781,0.2337,0,2017
4,2017,1,1,2,30,0,2.88,3.76,63.35,56.46,...,318.9,325.1,-6.74,1.5,2.99,0.1955,0.1782,0.2336,0,2017


In [3]:
flux_dir = Path("宝天曼站通量数据（2017-2018）")
flux_frames = []
for year in years:
    file_path = flux_dir / f"{year}年宝天曼通量30分钟数据.xls"
    df_year = pd.read_excel(file_path)
    df_year = df_year.iloc[1:].reset_index(drop=True)
    df_year["来源年份"] = year
    flux_frames.append(df_year)
df_flux_30min = pd.concat(flux_frames, ignore_index=True)
df_flux_30min.head()

,年,月,日,时,分,秒,NEE,RE,GEE,LE,Hs,来源年份
0,2017,1,1,0,30,0,0.005246,0.006197,-0.00095,3.1008,-11.19,2017
1,2017,1,1,1,0,0,0.006007,0.006178,-0.00017,4.7708,-6.9303,2017
2,2017,1,1,1,30,0,0.00605,0.00605,0,3.6732,-20.058,2017
3,2017,1,1,2,0,0,0.005716,0.005943,-0.000227,3.6732,-20.058,2017
4,2017,1,1,2,30,0,0.005478,0.006041,-0.000563,3.6732,-20.058,2017


In [4]:
# 3. 高效按日期对齐气象与通量数据
import numpy as np
date_columns = ["年", "月", "日", "时", "分"]
def prepare_half_hourly(df):
    numeric = df[date_columns].apply(pd.to_numeric, errors="coerce")
    dated = pd.to_datetime({
        "year": numeric["年"],
        "month": numeric["月"],
        "day": numeric["日"],
        "hour": numeric["时"],
        "minute": numeric["分"],
    }, errors="coerce")
    return (
        df.assign(date_time=dated)
        .drop(columns=date_columns + ["秒"], errors="ignore")
        .dropna(subset=["date_time"])
    )
df_meteo_prepared = prepare_half_hourly(df_meteo_30min)
df_flux_prepared = prepare_half_hourly(df_flux_30min)
df_merged = pd.merge(
    df_meteo_prepared,
    df_flux_prepared,
    on=["date_time", "来源年份"],
    how="inner",
    suffixes=("_meteo", "_flux"),
)
df_merged.sort_values(["date_time", "来源年份"], inplace=True)
df_merged.head()

,近地面空气温度,冠层上方空气温度,近地面空气湿度,冠层上方空气湿度,近地面风速,冠层上方风速,风向,大气压,光合有效辐射,向下的短波辐射,...,二层土壤体积含水量,三层土壤体积含水量,降水量,来源年份,date_time,NEE,RE,GEE,LE,Hs
0,3.24,4.09,57.43,52.44,0.75,0.51,27.21,86.09,0,0.17,...,0.1779,0.2339,0,2017,2017-01-01 00:30:00,0.005246,0.006197,-0.00095,3.1008,-11.19
1,2.85,4.05,57.14,52.19,0.58,0.02,352.9,86.06,0,0.03,...,0.1779,0.2338,0,2017,2017-01-01 01:00:00,0.006007,0.006178,-0.00017,4.7708,-6.9303
2,2.83,3.78,56.24,52.98,0.38,0,332.78,86.04,0,0,...,0.178,0.2337,0,2017,2017-01-01 01:30:00,0.00605,0.00605,0,3.6732,-20.058
3,2.76,3.55,58.67,56.45,0.73,0,28.51,86.02,0,0.04,...,0.1781,0.2337,0,2017,2017-01-01 02:00:00,0.005716,0.005943,-0.000227,3.6732,-20.058
4,2.88,3.76,63.35,56.46,0.36,0,323.13,85.99,0,0.1,...,0.1782,0.2336,0,2017,2017-01-01 02:30:00,0.005478,0.006041,-0.000563,3.6732,-20.058


In [5]:
# 4. 筛选保留的字段并清除无关指标
columns_to_exclude = {"来源年份", "RE", "GEE", "LE", "Hs"}
suffixes = ("_meteo", "_flux")
drop_candidates = set()
for col in columns_to_exclude:
    drop_candidates.add(col)
    for suffix in suffixes:
        drop_candidates.add(f"{col}{suffix}")
# Excel 导入常见的未命名列也一并移除
drop_candidates.update(
    column for column in df_merged.columns if column.lower().startswith("unnamed")
)
columns_removed = [column for column in df_merged.columns if column in drop_candidates]
df_selected = df_merged.drop(columns=columns_removed, errors="ignore")
print("Removed columns:", columns_removed[:10],"..." if len(columns_removed) > 10 else "")
df_selected.head()

Removed columns: ['来源年份', 'RE', 'GEE', 'LE', 'Hs'] 


,近地面空气温度,冠层上方空气温度,近地面空气湿度,冠层上方空气湿度,近地面风速,冠层上方风速,风向,大气压,光合有效辐射,向下的短波辐射,...,向上的长波辐射,净辐射,一层土壤温度,二层土壤温度,一层土壤体积含水量,二层土壤体积含水量,三层土壤体积含水量,降水量,date_time,NEE
0,3.24,4.09,57.43,52.44,0.75,0.51,27.21,86.09,0,0.17,...,325.7,-13.76,1.45,2.99,0.1948,0.1779,0.2339,0,2017-01-01 00:30:00,0.005246
1,2.85,4.05,57.14,52.19,0.58,0.02,352.9,86.06,0,0.03,...,325.6,-10.34,1.46,2.99,0.1949,0.1779,0.2338,0,2017-01-01 01:00:00,0.006007
2,2.83,3.78,56.24,52.98,0.38,0,332.78,86.04,0,0,...,325.1,-8.48,1.47,2.99,0.1952,0.178,0.2337,0,2017-01-01 01:30:00,0.00605
3,2.76,3.55,58.67,56.45,0.73,0,28.51,86.02,0,0.04,...,324.7,-4.82,1.48,2.99,0.1954,0.1781,0.2337,0,2017-01-01 02:00:00,0.005716
4,2.88,3.76,63.35,56.46,0.36,0,323.13,85.99,0,0.1,...,325.1,-6.74,1.5,2.99,0.1955,0.1782,0.2336,0,2017-01-01 02:30:00,0.005478


In [6]:
# 统计各列缺失/异常值（NaN、-9999、字符串NA等）并导出
from pathlib import Path
import pandas as pd
import numpy as np

if 'df_selected' not in globals():
    raise RuntimeError("需要已生成的 df_selected（含各列数据）。请先运行上方清洗流程。")

df = df_selected.copy()
total_rows = len(df)
na_tokens = {"na", "n/a", "nan", "null", "none", ""}
summary_rows = []

for col in df.columns:
    s = df[col]
    # NaN（pandas 缺失）
    mask_nan = s.isna()
    nan_count = int(mask_nan.sum())
    # -9999（数值或可转为数值的字符串）
    s_num = pd.to_numeric(s, errors="coerce")
    mask_minus9999 = s_num.eq(-99999)
    minus9999_count = int(mask_minus9999.sum())
    # 字符串形式的 NA（大小写及前后空白不敏感）
    s_str = s.astype(str).str.strip().str.lower()
    mask_na_str = s_str.isin(na_tokens)
    na_str_count = int(mask_na_str.sum())
    # 汇总“问题值”（三者并集）
    mask_problem = mask_nan | mask_minus9999 | mask_na_str
    problem_count = int(mask_problem.sum())
    # 统计行
    summary_rows.append({
        "column": col,
        "total_rows": total_rows,
        "nan_count": nan_count,
        "nan_rate": round(nan_count / total_rows, 6) if total_rows else 0.0,
        "minus9999_count": minus9999_count,
        "minus9999_rate": round(minus9999_count / total_rows, 6) if total_rows else 0.0,
        "na_string_count": na_str_count,
        "na_string_rate": round(na_str_count / total_rows, 6) if total_rows else 0.0,
        "problem_count": problem_count,
        "problem_rate": round(problem_count / total_rows, 6) if total_rows else 0.0,
    })

summary_df = pd.DataFrame(summary_rows)
summary_df = summary_df.sort_values(by=["problem_rate", "nan_rate", "minus9999_rate", "na_string_rate"], ascending=False).reset_index(drop=True)
display(summary_df)

# 导出为 Excel 和 CSV
# excel_path = Path("missing_summary.xlsx")
# csv_path = Path("missing_summary.csv")
# summary_df.to_excel(excel_path, index=False)
# summary_df.to_csv(csv_path, index=False)
# print("已导出缺失统计:", excel_path, "和", csv_path)

,column,total_rows,nan_count,nan_rate,minus9999_count,minus9999_rate,na_string_count,na_string_rate,problem_count,problem_rate
0,大气压,35040,0,0.000000,2553,0.072860,0,0.000000,2553,0.072860
1,向下的长波辐射,35040,0,0.000000,2535,0.072346,0,0.000000,2535,0.072346
2,向上的长波辐射,35040,0,0.000000,2535,0.072346,0,0.000000,2535,0.072346
3,向下的短波辐射,35040,0,0.000000,2527,0.072118,0,0.000000,2527,0.072118
4,向上的短波辐射,35040,0,0.000000,2527,0.072118,0,0.000000,2527,0.072118
5,净辐射,35040,0,0.000000,2527,0.072118,0,0.000000,2527,0.072118
6,冠层上方空气湿度,35040,0,0.000000,939,0.026798,0,0.000000,939,0.026798
7,近地面空气温度,35040,8,0.000228,0,0.000000,8,0.000228,8,0.000228
8,冠层上方空气温度,35040,8,0.000228,0,0.000000,8,0.000228,8,0.000228
9,近地面风速,35040,8,0.000228,0,0.000000,8,0.000228,8,0.000228


In [7]:
# 在所有数值列中去除包含-99999的行
value_cols = [c for c in df_selected.columns if c != "date"]
mask_minus99999 = df_selected[value_cols].apply(lambda col: pd.to_numeric(col, errors="coerce").eq(-99999), axis=0).any(axis=1)
rows_before = len(df_selected)
df_selected = df_selected[~mask_minus99999].copy()
rows_removed = rows_before - len(df_selected)
print(f"剔除含 -99999 的行数: {rows_removed}")
df_selected.head()

剔除含 -99999 的行数: 3520


,近地面空气温度,冠层上方空气温度,近地面空气湿度,冠层上方空气湿度,近地面风速,冠层上方风速,风向,大气压,光合有效辐射,向下的短波辐射,...,向上的长波辐射,净辐射,一层土壤温度,二层土壤温度,一层土壤体积含水量,二层土壤体积含水量,三层土壤体积含水量,降水量,date_time,NEE
0,3.24,4.09,57.43,52.44,0.75,0.51,27.21,86.09,0,0.17,...,325.7,-13.76,1.45,2.99,0.1948,0.1779,0.2339,0,2017-01-01 00:30:00,0.005246
1,2.85,4.05,57.14,52.19,0.58,0.02,352.9,86.06,0,0.03,...,325.6,-10.34,1.46,2.99,0.1949,0.1779,0.2338,0,2017-01-01 01:00:00,0.006007
2,2.83,3.78,56.24,52.98,0.38,0,332.78,86.04,0,0,...,325.1,-8.48,1.47,2.99,0.1952,0.178,0.2337,0,2017-01-01 01:30:00,0.00605
3,2.76,3.55,58.67,56.45,0.73,0,28.51,86.02,0,0.04,...,324.7,-4.82,1.48,2.99,0.1954,0.1781,0.2337,0,2017-01-01 02:00:00,0.005716
4,2.88,3.76,63.35,56.46,0.36,0,323.13,85.99,0,0.1,...,325.1,-6.74,1.5,2.99,0.1955,0.1782,0.2336,0,2017-01-01 02:30:00,0.005478


In [8]:
temp_cols = ["一层土壤温度", "二层土壤温度"]
extra_temp_cols = ["四层土壤温度", "五层土壤温度"]
moisture_cols = ["一层土壤体积含水量", "二层土壤体积含水量", "三层土壤体积含水量"]
for column_group, new_name in [
    (temp_cols, "土壤温度"),
    (moisture_cols, "土壤含水量"),
]:
    existing = [column for column in column_group if column in df_selected.columns]
    if not existing:
        print(f"Warning: 未找到 {new_name} 所需的列，跳过。")
        continue
    averaged = (
        df_selected[existing]
        .apply(pd.to_numeric, errors="coerce")
        .mean(axis=1)
    )
    df_selected[new_name] = averaged
    df_selected = df_selected.drop(columns=existing, errors="ignore")
df_selected = df_selected.drop(columns=extra_temp_cols, errors="ignore")
print("已生成列:", [name for _, name in [(temp_cols, "土壤温度"), (moisture_cols, "土壤含水量")]])
df_selected.head()

已生成列: ['土壤温度', '土壤含水量']


,近地面空气温度,冠层上方空气温度,近地面空气湿度,冠层上方空气湿度,近地面风速,冠层上方风速,风向,大气压,光合有效辐射,向下的短波辐射,向上的短波辐射,向下的长波辐射,向上的长波辐射,净辐射,降水量,date_time,NEE,土壤温度,土壤含水量
0,3.24,4.09,57.43,52.44,0.75,0.51,27.21,86.09,0,0.17,1.03,312.8,325.7,-13.76,0,2017-01-01 00:30:00,0.005246,2.220,0.202200
1,2.85,4.05,57.14,52.19,0.58,0.02,352.9,86.06,0,0.03,0.69,316,325.6,-10.34,0,2017-01-01 01:00:00,0.006007,2.225,0.202200
2,2.83,3.78,56.24,52.98,0.38,0,332.78,86.04,0,0,0.39,317,325.1,-8.48,0,2017-01-01 01:30:00,0.00605,2.230,0.202300
3,2.76,3.55,58.67,56.45,0.73,0,28.51,86.02,0,0.04,0.37,320.2,324.7,-4.82,0,2017-01-01 02:00:00,0.005716,2.235,0.202400
4,2.88,3.76,63.35,56.46,0.36,0,323.13,85.99,0,0.1,0.63,318.9,325.1,-6.74,0,2017-01-01 02:30:00,0.005478,2.245,0.202433


In [9]:
# 6. 调整列顺序：保留完整时间戳（含时分）
if "date_time" not in df_selected.columns:
    raise KeyError("df_selected 中缺少 date_time 列，无法生成 date")
df_selected["date"] = pd.to_datetime(df_selected["date_time"], errors="coerce")
df_selected = df_selected.drop(columns=["date_time"], errors="ignore")
priority_columns = ["date"]
trailing_columns = ["NEE"]
existing_priority = [column for column in priority_columns if column in df_selected.columns]
existing_trailing = [column for column in trailing_columns if column in df_selected.columns]
middle_columns = [
    column
    for column in df_selected.columns
    if column not in existing_priority + existing_trailing
]
new_order = existing_priority + middle_columns + existing_trailing
df_selected = df_selected.reindex(columns=new_order)
df_selected.head()

,date,近地面空气温度,冠层上方空气温度,近地面空气湿度,冠层上方空气湿度,近地面风速,冠层上方风速,风向,大气压,光合有效辐射,向下的短波辐射,向上的短波辐射,向下的长波辐射,向上的长波辐射,净辐射,降水量,土壤温度,土壤含水量,NEE
0,2017-01-01 00:30:00,3.24,4.09,57.43,52.44,0.75,0.51,27.21,86.09,0,0.17,1.03,312.8,325.7,-13.76,0,2.220,0.202200,0.005246
1,2017-01-01 01:00:00,2.85,4.05,57.14,52.19,0.58,0.02,352.9,86.06,0,0.03,0.69,316,325.6,-10.34,0,2.225,0.202200,0.006007
2,2017-01-01 01:30:00,2.83,3.78,56.24,52.98,0.38,0,332.78,86.04,0,0,0.39,317,325.1,-8.48,0,2.230,0.202300,0.00605
3,2017-01-01 02:00:00,2.76,3.55,58.67,56.45,0.73,0,28.51,86.02,0,0.04,0.37,320.2,324.7,-4.82,0,2.235,0.202400,0.005716
4,2017-01-01 02:30:00,2.88,3.76,63.35,56.46,0.36,0,323.13,85.99,0,0.1,0.63,318.9,325.1,-6.74,0,2.245,0.202433,0.005478


In [ ]:
# # 8. 基于互信息与树模型寻找非线性关系（按指定列顺序）
# from sklearn.feature_selection import mutual_info_regression
# from sklearn.ensemble import RandomForestRegressor
# from sklearn.model_selection import train_test_split
# candidate_columns = [
#     "近地面空气温度",
#     "冠层上方空气温度",
#     "近地面空气湿度",
#     "冠层上方空气湿度",
#     "近地面水汽压",
#     "冠层上方水汽压",
#     "近地面风速",
#     "冠层上方风速",
#     "风向",
#     "大气压",
#     "太阳辐射",
#     "净辐射",
#     "光合有效辐射",
#     "H",
#     "土壤温度",
#     "土壤含水量",
# ]
# target_column = "NEE"
# if target_column not in df_selected.columns:
#     raise ValueError("df_selected 中缺少 NEE 列，无法执行非线性分析。")
# available_features = [
#     column
#     for column in candidate_columns
#     if column in df_selected.columns
# ]
# if not available_features:
#     raise ValueError("候选特征均未在 df_selected 中找到，请检查列名。")
# X_numeric = df_selected[available_features].apply(pd.to_numeric, errors="coerce")
# valid_features = [column for column in X_numeric.columns if not X_numeric[column].isna().all()]
# if not valid_features:
#     raise ValueError("候选特征均为空，无法计算非线性关系。")
# analysis_frame = pd.concat([
#     df_selected[[target_column]].apply(pd.to_numeric, errors="coerce"),
#     X_numeric[valid_features],
# ], axis=1).dropna()
# if analysis_frame.empty:
#     raise ValueError("非线性分析需要的列为空，请检查缺失值或列名。")
# X = analysis_frame[valid_features]
# y = analysis_frame[target_column]
# mi_scores = mutual_info_regression(X, y, random_state=42)
# mi_series = (
#     pd.Series(mi_scores, index=valid_features)
#     .sort_values(ascending=False)
# )
# print("互信息得分 (完整列表):")
# print(mi_series.to_string())
# top_mi_features = mi_series.head(6).index.tolist()
# rf = RandomForestRegressor(
#     n_estimators=500,
#     max_depth=None,
#     random_state=42,
#     n_jobs=-1,
# )
# X_train, X_test, y_train, y_test = train_test_split(
#     X, y, test_size=0.2, shuffle=True, random_state=42
# )
# rf.fit(X_train, y_train)
# rf_importances = pd.Series(
#     rf.feature_importances_, index=valid_features
# ).sort_values(ascending=False)
# print("随机森林特征重要度 (完整列表):")
# print(rf_importances.to_string())

In [12]:
# 9. 输出协变量两两相关并按 |corr| 从高到低排序，并导出为Excel
from pathlib import Path
feature_candidates = [
    column
    for column in df_selected.columns
    if column not in {"date", "NEE"}
 ]
X_numeric = df_selected[feature_candidates].apply(pd.to_numeric, errors="coerce")
feature_columns = [
    column
    for column in X_numeric.columns
    if not X_numeric[column].isna().all()
 ]
if not feature_columns:
    raise ValueError("未找到可用于计算相关性的协变量。")
X = X_numeric[feature_columns]
corr_matrix = X.corr()
pairs = []
for row in corr_matrix.index:
    for col in corr_matrix.columns:
        if row >= col:
            continue
        pairs.append((row, col, corr_matrix.loc[row, col]))
correlation_df = pd.DataFrame(
    pairs,
    columns=["feature_1", "feature_2", "pearson_corr"],
)
correlation_df["abs_corr"] = correlation_df["pearson_corr"].abs()
correlation_df = correlation_df.sort_values(
    by="abs_corr", ascending=False
)
display(correlation_df.drop(columns=["abs_corr"]))

# 导出为 Excel（去除辅助列 abs_corr），保存到当前目录
export_df = correlation_df.drop(columns=["abs_corr"])
out_path = Path("correlations.xlsx")
export_df.to_excel(out_path, index=False)


,feature_1,feature_2,pearson_corr
115,净辐射,向下的短波辐射,0.989610
64,光合有效辐射,向下的短波辐射,0.987991
4,冠层上方空气温度,近地面空气温度,0.981976
14,冠层上方空气温度,向上的长波辐射,0.981509
68,光合有效辐射,净辐射,0.979318
...,...,...,...
26,冠层上方空气湿度,风向,-0.010049
126,土壤温度,风向,0.008524
37,近地面风速,降水量,0.003499
42,冠层上方风速,大气压,-0.003196


In [14]:
# 9. 删除指定的协变量列
columns_to_remove = [
    '向下的短波辐射',
    '向上的长波辐射',
    "太阳辐射",
    "光合有效辐射",
    "冠层上方空气湿度",
    "近地面风速",
    "降水量",
    '土壤温度',
    "冠层上方空气温度",
    "冠层上方水汽压",
]
existing_to_remove = [column for column in columns_to_remove if column in df_selected.columns]
if existing_to_remove:
    df_selected = df_selected.drop(columns=existing_to_remove)
    print("已删除列:", existing_to_remove)
else:
    print("未找到需要删除的列。")
df_selected.head()

已删除列: ['土壤温度']


,date,近地面空气温度,近地面空气湿度,冠层上方风速,风向,大气压,向上的短波辐射,向下的长波辐射,净辐射,土壤含水量,NEE
0,2017-01-01 00:30:00,3.24,57.43,0.51,27.21,86.09,1.03,312.8,-13.76,0.202200,0.005246
1,2017-01-01 01:00:00,2.85,57.14,0.02,352.9,86.06,0.69,316,-10.34,0.202200,0.006007
2,2017-01-01 01:30:00,2.83,56.24,0,332.78,86.04,0.39,317,-8.48,0.202300,0.00605
3,2017-01-01 02:00:00,2.76,58.67,0,28.51,86.02,0.37,320.2,-4.82,0.202400,0.005716
4,2017-01-01 02:30:00,2.88,63.35,0,323.13,85.99,0.63,318.9,-6.74,0.202433,0.005478


In [15]:
# 10. 仅执行列重命名（不做数值转换、插值或缺失填充）
col_map = {
    "date": "date",
    "近地面空气温度": "Ta",
    "近地面空气湿度": "RH",
    "近地面水汽压": "VPD",
    "冠层上方风速": "WS",
    "风向": "WD",
    "大气压": "PA",
    "净辐射": "Rn",
    "土壤温度": "Ts",
    "土壤含水量": "SWC",
    "NEE": "NEE",
    "向上的短波辐射": "SWout",
    "向下的长波辐射": "LWin",
}									
existing_map = {old: new for old, new in col_map.items() if old in df_selected.columns}
df_selected = df_selected.rename(columns=existing_map)
print("已重命名列:", existing_map)
df_selected.head()

已重命名列: {'date': 'date', '近地面空气温度': 'Ta', '近地面空气湿度': 'RH', '冠层上方风速': 'WS', '风向': 'WD', '大气压': 'PA', '净辐射': 'Rn', '土壤含水量': 'SWC', 'NEE': 'NEE', '向上的短波辐射': 'SWout', '向下的长波辐射': 'LWin'}


,date,Ta,RH,WS,WD,PA,SWout,LWin,Rn,SWC,NEE
0,2017-01-01 00:30:00,3.24,57.43,0.51,27.21,86.09,1.03,312.8,-13.76,0.202200,0.005246
1,2017-01-01 01:00:00,2.85,57.14,0.02,352.9,86.06,0.69,316,-10.34,0.202200,0.006007
2,2017-01-01 01:30:00,2.83,56.24,0,332.78,86.04,0.39,317,-8.48,0.202300,0.00605
3,2017-01-01 02:00:00,2.76,58.67,0,28.51,86.02,0.37,320.2,-4.82,0.202400,0.005716
4,2017-01-01 02:30:00,2.88,63.35,0,323.13,85.99,0.63,318.9,-6.74,0.202433,0.005478


In [16]:
# 对除 NEE 外的数值列保留4位小数，NEE保留8位；剔除含科学计数法(e/E)的行
from pandas.api.types import is_numeric_dtype
import pandas as pd
import numpy as np

if 'df_selected' not in globals():
    raise RuntimeError("需要已生成的 df_selected。")

# 1) 剔除含 e/E 的行（排除日期列）
value_cols = [c for c in df_selected.columns if c != "date"]
mask_e_any = df_selected[value_cols].apply(
    lambda col: col.astype(str).str.contains(r"[eE]", regex=True, na=False)
).any(axis=1)
rows_before = len(df_selected)
df_selected = df_selected[~mask_e_any].copy()
rows_removed = rows_before - len(df_selected)
print(f"剔除含 e/E 科学计数法的行数: {rows_removed}")

# 2) 统一数值化后再四舍五入：非NEE保留4位，NEE保留8位
for col in value_cols:
    # 将可转换的内容转为数值（无法转换的设为 NaN）
    df_selected[col] = pd.to_numeric(df_selected[col], errors="coerce")
    # 仅对数值列执行四舍五入
    if is_numeric_dtype(df_selected[col]):
        if col == "NEE":
            df_selected[col] = df_selected[col].round(8)
        else:
            df_selected[col] = df_selected[col].round(4)
print("完成小数位处理：非NEE→4位，NEE→8位。")
df_selected.head()

剔除含 e/E 科学计数法的行数: 8
完成小数位处理：非NEE→4位，NEE→8位。


,date,Ta,RH,WS,WD,PA,SWout,LWin,Rn,SWC,NEE
0,2017-01-01 00:30:00,3.24,57.43,0.51,27.21,86.09,1.03,312.8,-13.76,0.2022,0.005246
1,2017-01-01 01:00:00,2.85,57.14,0.02,352.90,86.06,0.69,316.0,-10.34,0.2022,0.006007
2,2017-01-01 01:30:00,2.83,56.24,0.00,332.78,86.04,0.39,317.0,-8.48,0.2023,0.006050
3,2017-01-01 02:00:00,2.76,58.67,0.00,28.51,86.02,0.37,320.2,-4.82,0.2024,0.005716
4,2017-01-01 02:30:00,2.88,63.35,0.00,323.13,85.99,0.63,318.9,-6.74,0.2024,0.005478


In [17]:
# 11. 导出处理后的数据集
from pathlib import Path
output_path = Path("BaoTianMan.csv")
df_selected.to_csv(output_path, index=False)
print("已导出:", output_path)

已导出: BaoTianMan.csv
